In [1]:
# Frequency-only (FFT magnitude)
import os, cv2, numpy as np, random
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
SEED=15
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

In [10]:
REAL_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/train"
FAKE_DIR = "/content/drive/My Drive/Research Project/StyleGAN2_7k/train"
IMG_SIZE=(128,128)

In [11]:
def load_fft(real_dir, fake_dir, img_size=(128,128), limit=None):
    X, y = [], []
    def proc(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0
        F = np.fft.fft2(img)
        Fshift = np.fft.fftshift(F)
        mag = np.abs(Fshift)
        mag_log = np.log(mag + 1e-8)

        return mag_log[..., None]
    reals = sorted(os.listdir(real_dir))
    fakes = sorted(os.listdir(fake_dir))
    if limit: reals, fakes = reals[:limit], fakes[:limit]
    for fn in reals: X.append(proc(os.path.join(real_dir, fn))); y.append(0)
    for fn in fakes: X.append(proc(os.path.join(fake_dir, fn))); y.append(1)
    return np.array(X), np.array(y)

In [12]:
X, y = load_fft(REAL_DIR, FAKE_DIR)
print("FFT shape:", X.shape, "labels:", y.shape)

FFT shape: (10575, 128, 128, 1) labels: (10575,)


In [13]:
def build_branch(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(16,(4,4),activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64,activation='relu')(x)
    out = layers.Dense(1,activation='sigmoid')(x)
    return models.Model(inp,out)

In [14]:
model = build_branch((128,128,1))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [15]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
early_stop = EarlyStopping(monitor='val_loss', patience=9, restore_best_weights=True)
history = model.fit(Xtr, ytr, validation_data=(Xte,yte), epochs=20, batch_size=35, callbacks=[early_stop], verbose=1)

print("Test eval:", model.evaluate(Xte, yte, verbose=0))

Epoch 1/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 0.7746 - loss: 0.4053 - val_accuracy: 0.9896 - val_loss: 0.0394
Epoch 2/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9789 - loss: 0.0698 - val_accuracy: 0.9887 - val_loss: 0.0284
Epoch 3/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9818 - loss: 0.0594 - val_accuracy: 0.9953 - val_loss: 0.0151
Epoch 4/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9866 - loss: 0.0476 - val_accuracy: 0.9972 - val_loss: 0.0105
Epoch 5/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9887 - loss: 0.0432 - val_accuracy: 0.9976 - val_loss: 0.0080
Epoch 6/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9913 - loss: 0.0296 - val_accuracy: 0.9972 - val_loss: 0.0083
Epoch 7/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9907 - loss: 0.0290 - val_accuracy: 0.9986 - val_loss: 0.0072
Epoch 8/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9941 - loss: 0.0217 - val_acc

In [16]:
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

REAL_TEST_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/test"
FAKE_TEST_DIR = "/content/drive/My Drive/Research Project/StyleGAN3_7k/test"

X_test,  y_test = load_fft(REAL_TEST_DIR, FAKE_TEST_DIR, img_size=IMG_SIZE, limit=None)

print("StyleGAN3 Test Shapes:", X_test.shape, y_test.shape)


test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print("StyleGAN3 Test Loss:", test_loss)
print("StyleGAN3 Test Accuracy:", test_acc)


y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

print("\nClassification Report (StyleGAN3 Test Set):")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
StyleGAN3 Test Shapes: (3525, 128, 128, 1) (3525,)
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9971 - loss: 0.0202
StyleGAN3 Test Loss: 0.015310184098780155
StyleGAN3 Test Accuracy: 0.9965957403182983
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step

Classification Report (StyleGAN3 Test Set):
              precision    recall  f1-score   support

        Real       0.99      1.00      1.00      1750
        Fake       1.00      0.99      1.00      1775

    accuracy                           1.00      3525
   macro avg       1.00      1.00      1.00      3525
weighted avg       1.00      1.00      1.00      3525

